# Phase 2: Data Exploration
## Porto Weather Dataset - Minimum Temperature Analysis

**Project:** AA Project 3 - Approximate Counting and Frequent Items  
**Author:** João Roldão (113920)  
**Date:** December 2024

---

This notebook performs comprehensive exploratory data analysis on the Porto minimum temperature dataset as outlined in `development_plan.md` Phase 2.

**Objectives:**
- Load and validate the porto.csv dataset
- Calculate summary statistics
- Identify the 31 unique temperature values
- Create frequency distributions
- Visualize temperature distribution
- Check for data quality issues
- Save exploratory figures

In [ ]:
# Import required libraries
import sys
sys.path.append('..')  # Add parent directory to path

from src.utils.data_loader import load_porto_temperatures, get_exact_frequencies
from src.utils.config import FIGURES_EXPLORATORY_PATH

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Ensure figures directory exists
Path(FIGURES_EXPLORATORY_PATH).mkdir(parents=True, exist_ok=True)

print("✓ Imports successful")
print(f"✓ Figures will be saved to: {FIGURES_EXPLORATORY_PATH}")

## 1. Load Dataset

In [ ]:
# Load temperature data
temps = load_porto_temperatures(validate=True, verbose=True)

## 2. Summary Statistics

In [ ]:
# Calculate comprehensive statistics
print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
print(f"Total observations: {len(temps):,}")
print(f"Unique temperatures: {temps.nunique()}")
print(f"Temperature range: [{temps.min():.1f}°C, {temps.max():.1f}°C]")
print(f"Mean temperature: {temps.mean():.2f}°C")
print(f"Median temperature: {temps.median():.2f}°C")
print(f"Std deviation: {temps.std():.2f}°C")
print("=" * 60)

# Display pandas describe
print("\nDetailed Statistics:")
print(temps.describe())

## 3. Frequency Distribution

**Critical insight:** With only 31 unique temperatures, Space-Saving with k ≥ 31 will capture everything perfectly.

In [ ]:
# Get exact frequency distribution
freqs = get_exact_frequencies()

print("=" * 60)
print("FREQUENCY DISTRIBUTION (All 31 unique temperatures)")
print("=" * 60)
print(freqs)
print("=" * 60)

# Save to CSV for later use
freqs_df = freqs.reset_index()
freqs_df.columns = ['temperature', 'count']
freqs_df.to_csv('../data/processed/exact_frequencies.csv', index=False)
print("✓ Saved exact frequencies to data/processed/exact_frequencies.csv")

## 4. Top-N Analysis

In [ ]:
# Analyze top-5, top-10, top-15, top-20
print("=" * 60)
print("TOP-N ANALYSIS")
print("=" * 60)

for n in [5, 10, 15, 20]:
    top_n = freqs.head(n)
    coverage = (top_n.sum() / freqs.sum()) * 100
    print(f"\nTop-{n} temperatures:")
    print(f"  Coverage: {coverage:.2f}% of all observations")
    print(f"  Temperatures: {list(top_n.index)}")

print("=" * 60)

## 5. Visualizations

In [ ]:
# Histogram of temperature distribution
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(temps, bins=31, edgecolor='black', alpha=0.7, color='steelblue')
ax.set_xlabel('Minimum Temperature (°C)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of Minimum Temperatures in Porto', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURES_EXPLORATORY_PATH}/temperature_histogram.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved histogram")

In [ ]:
# Bar chart of top 20 frequencies
fig, ax = plt.subplots(figsize=(14, 6))
top_20 = freqs.head(20)
bars = ax.bar(range(len(top_20)), top_20.values, color='coral', edgecolor='black')
ax.set_xticks(range(len(top_20)))
ax.set_xticklabels([f"{temp:.1f}°C" for temp in top_20.index], rotation=45, ha='right')
ax.set_xlabel('Temperature', fontsize=12)
ax.set_ylabel('Frequency (Count)', fontsize=12)
ax.set_title('Top 20 Most Frequent Minimum Temperatures', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIGURES_EXPLORATORY_PATH}/top20_frequencies.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved top-20 bar chart")

In [ ]:
# Cumulative percentage plot
cumulative_pct = (freqs.cumsum() / freqs.sum()) * 100

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(range(1, len(cumulative_pct) + 1), cumulative_pct, 
        marker='o', linewidth=2, markersize=6, color='darkgreen')
ax.axhline(y=80, color='red', linestyle='--', alpha=0.7, label='80% coverage')
ax.axhline(y=90, color='orange', linestyle='--', alpha=0.7, label='90% coverage')
ax.set_xlabel('Number of Unique Temperatures (Ranked by Frequency)', fontsize=12)
ax.set_ylabel('Cumulative Coverage (%)', fontsize=12)
ax.set_title('Cumulative Coverage of Temperatures', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_EXPLORATORY_PATH}/cumulative_coverage.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved cumulative coverage plot")

## 6. Data Quality Checks

In [ ]:
# Load full dataset to check for missing values
df_full = pd.read_csv('../data/raw/porto.csv')

print("=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)
print(f"Total rows in dataset: {len(df_full):,}")
print(f"Missing values in mintempC: {df_full['mintempC'].isna().sum()}")
print(f"Data type: {df_full['mintempC'].dtype}")
print(f"\nTemperature value check:")
print(f"  All values are numeric: {pd.api.types.is_numeric_dtype(df_full['mintempC'])}")
print(f"  Min value plausible: {temps.min()} °C (✓ if 0-25°C range)")
print(f"  Max value plausible: {temps.max()} °C (✓ if 0-25°C range)")

# Check for outliers (Mediterranean climate: expect 0-25°C)
outliers = temps[(temps < 0) | (temps > 25)]
if len(outliers) > 0:
    print(f"\n⚠ WARNING: {len(outliers)} potential outliers outside 0-25°C range:")
    print(outliers.value_counts())
else:
    print(f"\n✓ No outliers detected (all temps in 0-25°C range)")

print("=" * 60)

## 7. Key Insights for Algorithm Implementation

### Critical Findings:
1. **31 unique temperatures** → Space-Saving with k ≥ 31 captures everything
2. Distribution characteristics inform parameter choices:
   - Test k = {10, 20, 30} for approximation behavior
   - Test k = {40, 50} for perfect capture verification
3. Fixed Probability Counter expected behavior:
   - High-frequency items (top 10): ~2-3% relative error
   - Low-frequency items: Higher variance, ~10-30% relative error
   
### Next Steps:
- **Phase 3:** Implement Exact Counter
- **Phase 4:** Implement Fixed Probability Counter (p=0.25)
- **Phase 5:** Implement Space-Saving Algorithm
- **Phase 6:** Comprehensive comparison

In [ ]:
# Summary for report
print("=" * 60)
print("PHASE 2 COMPLETE - DATA EXPLORATION SUMMARY")
print("=" * 60)
print(f"✓ Dataset validated: 3,946 observations")
print(f"✓ Identified 31 unique temperature values")
print(f"✓ Generated 3 exploratory figures")
print(f"✓ No data quality issues detected")
print(f"✓ Ready for Phase 3: Algorithm Implementation")
print("=" * 60)